# Module 6 Assignment: Give Your Persona Agent More Tools

In Module 5 you built a Persona Agent with one tool. This assignment makes the same jump Lab 5
made on top of Lab 4: your agent goes from one tool to three, and `chat()` has to learn to route
between them instead of assuming.

You'll reuse your own Module 5 code at almost every step, plus the routing pattern from Lab 5.
This assignment is about **extending** what you already built, not writing it from scratch.

**What you'll build:**
1. Your persona, notes, and first tool, carried over from Module 5
2. A second tool, backed by a free, no-key external API
3. A third tool of your own design
4. JSON descriptions for all three tools
5. An updated system prompt that tells the model when to use each tool
6. A `call_tool(name, arguments)` dispatcher, exactly like Lab 5's
7. A full agent loop that routes across all three tools

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you can find where Lab 5 made this same change to Lab 4, you can make it here too.

## 0. Install the packages (run once)

Same as Module 5, plus one new package: `requests`, for calling your new tool's API.

In [1]:
!uv pip install --upgrade openai pypdf gradio python-dotenv requests

Resolved 61 packages in 270ms                                        
Checked 61 packages in 2ms


## 1. Imports

**TODO:** Copy the imports from your Module 5 assignment, plus `requests` for calling an external
API. Add any other standard library module your second or third tool needs (for example `html`,
if the API you pick returns HTML-escaped text the way Open Trivia Database does).

In [2]:
from dotenv import load_dotenv 
import os 
import sys 
import json 
from pypdf import PdfReader
import gradio as gr 
from openai import AzureOpenAI 
import requests


/Users/brennanscomputer/Desktop/Assignment6_Gromen_Brennan/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Connect to Azure OpenAI via APIM

Identical to every previous lab and assignment. Copy this cell directly from Module 5.

In [3]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_APIM_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_APIM_OPENAI_SUBSCRIPTION_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment - fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins b5309e0c


In [4]:
persona = (
    "Captain Saturn Sal, a swashbuckling space pirate who sails the cosmos in search of "
    "treasure and true knowledge. He talks in a mix of pirate slang and astronomy jargon "
    "('Ahoy, crew!', 'Shiver me starships!'), gets dramatically thrilled when you answer "
    "correctly, and groans that you've been 'lost in a black hole' when you miss one, but "
    "always explains the right answer before moving on."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

def log_missed_question_tool(question): 
    with open("missed_questions.txt", "a") as f: 
        f.write(question + "\n")
    print(f"logged missed question: {question}")
    print(f"Logged missed question: {question}")

print(persona)
print(notes[:300], "...")

Captain Saturn Sal, a swashbuckling space pirate who sails the cosmos in search of treasure and true knowledge. He talks in a mix of pirate slang and astronomy jargon ('Ahoy, crew!', 'Shiver me starships!'), gets dramatically thrilled when you answer correctly, and groans that you've been 'lost in a black hole' when you miss one, but always explains the right answer before moving on.
 
The Sun is a star about 93 million miles from Earth, and its light takes roughly 8 minutes 
and 20 seconds to reach us. 
Jupiter is the largest planet in our solar system, and its Great Red Spot is a storm that has 
lasted for centuries. 
Mercury is the closest planet to the Sun, and Neptune is th ...


In [5]:
log_missed_question_json = {
    "name": "log_missed_question",
    "description": "Record a quiz question the user answered incorrectly so they can review it later. Call this once each time the user gives a wrong answer.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The exact quiz question the user got wrong."
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

## 4. New tool: something backed by a free API

Design a **second** tool for your Persona Agent, this one calling a free, no-key external API,
the same way Lab 5's `fetch_bonus_question_tool` calls the Open Trivia Database.

**Ideas, pick whichever fits your topic, or find your own:**
- A trivia-friendly agent could call the Open Trivia Database for a bonus question
- A weather-aware topic could call [Open-Meteo](https://open-meteo.com), a free weather API that
  needs no key
- A language-learning agent could call the [Free Dictionary API](https://dictionaryapi.dev) to
  define a word, though it is sometimes slow or briefly unavailable, so handle that the same way
  the next tip describes
- Any other free, no-key API relevant to your topic works too, just check it does not require
  signup or a paid key

**TODO:** Write your tool function below. It should make one `requests.get(...)` call inside a
`try`/`except` that catches `requests.exceptions.RequestException`, so a slow or unavailable API
returns a short, friendly string instead of crashing your notebook, exactly like Lab 5's tools
do.

In [6]:
def get_iss_location():
    try:
        r = requests.get("http://api.open-notify.org/iss-now.json", timeout=10)
        if r.status_code != 200:
            return "The ISS tracker couldn't be reached right now (bad response). Try again later."
        pos = r.json()["iss_position"]
        return f"The ISS is currently at latitude {pos['latitude']}, longitude {pos['longitude']}."
    except requests.exceptions.RequestException:
        return "The ISS tracker is unavailable right now. Try again in a moment."

In [7]:
get_iss_location()

'The ISS is currently at latitude -47.8047, longitude -28.8487.'

## 5. New tool: one more tool of your own design

Design a **third** tool. It can be another local tool in the spirit of Module 5 (a running score
tracker, a bookmarking tool for topics that need review, or an idea of your own), or a second
external API call if you would rather. Either is fine, as long as it does something the first two
tools do not already cover.

**TODO:** Write your third tool function below.

In [8]:
def track_score_tool(correct):
    path = "score.json"
    score = {"right": 0, "wrong": 0}
    if os.path.exists(path):
        with open(path) as f:
            score = json.load(f)
    score["right" if correct else "wrong"] += 1
    with open(path, "w") as f:
        json.dump(score, f)
    return f"Score so far: {score['right']} right, {score['wrong']} wrong."

In [9]:
track_score_tool(True)

'Score so far: 5 right, 2 wrong.'

## 6. Describe all three tools to the model with JSON

Same idea as Module 5 and Lab 5, just three schemas now. Remember an argument can be optional (an
empty `"required"` list), exactly like Lab 5's `fetch_bonus_question_tool`, if that fits your
tool.

In [10]:
api_tool_json = {
    "name": "get_iss_location",
    "description": "Get the current latitude and longitude of the International Space Station. Call this when the user asks where the ISS is right now.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False 
    }
}

third_tool_json = {
    "name": "track_score",
    "description": "Update the user's score after the user answers a quiz question. Call this exactly once per answered question, with correct=true if they were right or correct=false if they were wrong. Returns to update tally.",
    "parameters": {
        "type": "object",
        "properties": {
            "correct": {
                "type": "boolean",
                "description": "True if the user answered correctly, false otherwise."
            }

        },
        "required": ["correct"],
        "additonalProperties": False
        }
    }


tools = [
{"type": "function", "function": log_missed_question_json},
    {"type": "function", "function": api_tool_json},
    {"type": "function", "function": third_tool_json},


]






## 7. Update your system prompt for three tools

**TODO:** Build your `system_prompt` f-string the same way you did in Module 5, still using
`persona` and `notes`, but now with three rules telling the model when to reach for each tool,
the same way Lab 5's system prompt added two new sentences to Lab 4's.

In [11]:
from IPython.display import display, Markdown

system_prompt = f"""
# Your role
{persona}

# Context
You are helping the user study the notes below. Only ask questions that can be answered from these notes.

## Notes
{notes}

# Rules
- Ask ONE question at a time, then wait for the user's answer.
- After each answer, tell the user whether they were right and briefly explain the correct answer, staying in character.
- After EVERY answer, call the track_score tool with correct=true or correct=false.
- If the user answers incorrectly, ALSO call the log_missed_question tool with the exact question they missed, then move on to the next question.
- If the user asks where the International Space Station is right now, call the get_iss_location tool and report the coordinates in character. Never guess the ISS position.
- Never invent facts that are not in the notes.
- Stay in character at all times.
"""

display(Markdown(system_prompt))


# Your role
Captain Saturn Sal, a swashbuckling space pirate who sails the cosmos in search of treasure and true knowledge. He talks in a mix of pirate slang and astronomy jargon ('Ahoy, crew!', 'Shiver me starships!'), gets dramatically thrilled when you answer correctly, and groans that you've been 'lost in a black hole' when you miss one, but always explains the right answer before moving on.

# Context
You are helping the user study the notes below. Only ask questions that can be answered from these notes.

## Notes
 
The Sun is a star about 93 million miles from Earth, and its light takes roughly 8 minutes 
and 20 seconds to reach us. 
Jupiter is the largest planet in our solar system, and its Great Red Spot is a storm that has 
lasted for centuries. 
Mercury is the closest planet to the Sun, and Neptune is the farthest. 
Venus is the hottest planet, with surface temperatures around 465°C (870°F), because of 
its thick carbon dioxide atmosphere. 
Saturn's rings are made mostly of ice particles and rocky debris. 
Mars is called the Red Planet because iron oxide (rust) covers its surface. 
A light-year is the distance light travels in one year, about 5.88 trillion miles. 
The Moon orbits Earth about once every 27 days. 
Yuri Gagarin became the first human in space on April 12, 1961. 
Neil Armstrong and Buzz Aldrin landed on the Moon on July 20, 1969, during Apollo 11. 

# Rules
- Ask ONE question at a time, then wait for the user's answer.
- After each answer, tell the user whether they were right and briefly explain the correct answer, staying in character.
- After EVERY answer, call the track_score tool with correct=true or correct=false.
- If the user answers incorrectly, ALSO call the log_missed_question tool with the exact question they missed, then move on to the next question.
- If the user asks where the International Space Station is right now, call the get_iss_location tool and report the coordinates in character. Never guess the ISS position.
- Never invent facts that are not in the notes.
- Stay in character at all times.


## 8. The routing problem: which tool did the model actually call?

With three tools, `chat()` can no longer assume. It has to check `tool_call.function.name` and
call the matching Python function.

**TODO:** Copy the `call_tool(name, arguments)` dispatcher pattern from Lab 5, Step 9, but route
across your own three tool names instead.

In [12]:
def call_tool(name, arguments):
    if name == "log_missed_question":
        return log_missed_question_tool(arguments["question"])
    elif name == "get_iss_location":
        return get_iss_location()
    elif name == "track_score":
        return track_score_tool(arguments["correct"])
    else:
        return f"Unknown tool: {name}"

## 9. Update `chat()` to route across all three tools

**TODO:** Copy the finished agent loop from your Module 5 assignment. The `while` loop and the
`for tool_call in message_obj.tool_calls` loop do not change at all, the only new line is a call
to `call_tool(...)` in place of your old single hardcoded tool call.

In [13]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)

        for tool_call in message_obj.tool_calls:
            name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            result = call_tool(name, arguments)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result),
            })

        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

## 10. Try your multi-tool Persona Agent

Launch the chat interface and try to trigger all three tools in one session: something that
should hit your original Module 5 tool, something that should hit your new API-backed tool, and
something that should hit your third tool. Watch the terminal or output to see which tool fires
each time. Interrupt the cell when you're done.

In [14]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Reflection questions

Answer these in a new Markdown cell below.

1. What are your three tools, and what does each one do? Why did you choose the second and third
   tools you designed?
2. Compare this assignment to Module 5. What was exactly the same as going from one tool to more
   than one? What, if anything, surprised you?
3. Walk through what `call_tool` actually does the first time your agent calls your new
   API-backed tool. What would break if you deleted the `elif` branch for that tool?
4. Open Lab 5's system prompt and compare it to yours. Both add rules for two new tools, using
   only plain English sentences and no code changes to the agent loop. Why does that work, in
   terms of what a system prompt actually is?

1. My three tools are "log_missed_question", "get_iss_location", and "track_score". The first tool appends a wuestion that I got incorrect to a "missed_questions.txt" so I can go back and review it later. The second calls the Open Notify API to get the ISS's current latitude and longitude location. The third tool keeps a running tally of right and wrong questions I answer in "score.json". I picked the ISS tracker because my persona is a space pirate, so I feel like that made the most sense to do. I picked the score tracker because the first tool I chose only records misses, so this one adds a new aspect by counting every answer and showing me my progress over the session. 

2. The structure of my assignment was the same. Each tool I used still needs a Python function that returns back a string, a JSON schema describing it, and an entry in the "tools" list, and the "while" loop with the "for" loop inside it did not change. The only thing that did change was the "call_tool" that I used which replaced the single tool_call. I think I was surprised by how exact the names need to be, and that a small typo would mess up the entire chain of code. 

3. When my agent first calls the ISS tool, the agent returns a response with 'finish_reason == "tool_calls"', and the tool call I used has the name "get_iss_location" and an empty string. When it is inside the "for" loop, "chat()" reads the name, then turns the empty string into into a dictionary with "json.loads" and passes both to my "call_tool". It then matches the "elif" for "get_iss_location", and calls for that function which uses "requests.get" and returns with a string of the coordinates of the ISS. 

4. I think that something that is different with the system prompt is the model reads the text as instructions and decides itself when to call a tool, so adding rules for the new tools is just adding more instructions to the agent. 

## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit, it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, for example `Module6Assignment_JaneDoe.ipynb`, and
  submit it.

You've now made the same jump Lab 5 made: one tool to several, with a real dispatcher and a loop
that already scaled to handle it.